In [5]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

# Week 3

**Objective**

Prepare the residential single-family housing dataset for a Linear Regression model by:

- Handling missing values
- Encoding categorical variables
- Scaling numerical features
- Creating a time-based train/test split
- Exporting a cleaned dataset

## Load Dataset

In [8]:
data_folder = "data"

csv_files = sorted(glob.glob(os.path.join(data_folder, "*.csv")))

dfs = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)
    dfs.append(df)

sold_df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(csv_files)} CSV files.")
print(sold_df.shape)

Loaded 30 CSV files.
(794271, 82)


In [9]:
sold_df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
0,"Carpet,Wood",True,NaN,NaN,True,98000.0,556366533,michellefsellsoc@gmail.com,2022-02-25,95000.0,...,0.0,ABC Unified,92708,0.0,NaN,NaN,False,False,NaN,NaN
1,NaN,False,NaN,NaN,False,1200.0,556366530,dineshcalre@gmail.com,2022-02-19,1200.0,...,1.0,Apple Valley Unified,92308,0.0,43000.0,NaN,False,False,NaN,NaN
2,NaN,True,NaN,NaN,False,1100000.0,556366044,cindydavishomes@gmail.com,2022-04-15,1100000.0,...,1.0,Solana Beach,92075,370.0,NaN,NaN,False,False,NaN,NaN
3,NaN,True,NaN,NaN,False,2499999.0,556365765,bryanmeathe@gmail.com,2022-01-04,2499999.0,...,2.0,Carlsbad Unified,92008,140.0,13376.0,NaN,False,False,NaN,NaN
4,"Carpet,Tile",NaN,NaN,NaN,NaN,598888.0,556365290,steven@westsideres.com,2022-01-12,640000.0,...,1.0,Other,95111,300.0,2738.0,NaN,False,False,NaN,NaN


## Filter Dataset

The project instructions specify that the prediction model should only be built
using residential single-family homes.

In addition, our team decided to use data beginning in June 2023 because earlier
months contain very few observations and are likely incomplete.

In [10]:
# Convert CloseDate to datetime
sold_df["CloseDate"] = pd.to_datetime(sold_df["CloseDate"])

# Residential Single Family only
sold_df = sold_df[
    (sold_df["PropertyType"] == "Residential") &
    (sold_df["PropertySubType"] == "SingleFamilyResidence")
]

# June 2023 onward
sold_df = sold_df[
    sold_df["CloseDate"] >= "2023-06-01"
]

print(f"Rows after filtering: {sold_df.shape[0]:,}")
print(f"Columns: {sold_df.shape[1]}")

Rows after filtering: 391,673
Columns: 82


In [11]:
sold_df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
542,"Carpet,Laminate",False,NaN,NaN,False,675000.0,554248930,roseb7@hotmail.com,2023-07-03,500000.0,...,NaN,NaN,90044,NaN,5988.0,NaN,False,False,NaN,NaN
955,"Laminate,SeeRemarks,Tile",False,NaN,NaN,False,315000.0,552093317,isabels.re4u@gmail.com,2023-10-27,331000.0,...,3.0,NaN,92392,NaN,7525.0,NaN,False,False,NaN,NaN
1057,"Carpet,Laminate,Tile,Wood",True,NaN,NaN,False,375000.0,552062457,LoniVogler@gmail.com,2023-07-05,410000.0,...,2.0,San Jacinto Unified,92582,0.0,7405.0,NaN,False,False,NaN,NaN
1421,NaN,False,NaN,NaN,False,484900.0,551932712,offers@riversidereo1.com,2023-07-19,519000.0,...,2.0,Compton Unified,90221,0.0,4764.0,NaN,False,False,NaN,NaN
2082,"Carpet,Tile,Wood",True,NaN,NaN,False,2740000.0,544556015,gerri@gerriwulffestates.com,2023-06-26,2999822.9,...,3.0,Bonita Unified,91773,671.0,22215.6,NaN,False,False,NaN,NaN


## Remove Invalid Close Prices

Rather than removing the lowest and highest 5% of observations, fixed price
thresholds were used.

Our team decided to:

- Remove sales below $50,000
- Remove sales above $5,000,000

These thresholds preserve legitimate luxury sales while removing obvious data
entry errors.

In [12]:
before = len(sold_df)

sold_df = sold_df[
    (sold_df["ClosePrice"] >= 50000) &
    (sold_df["ClosePrice"] <= 5000000)
]

removed = before - len(sold_df)

print(f"Removed {removed:,} records")
print(f"Remaining homes: {len(sold_df):,}")

Removed 6,176 records
Remaining homes: 385,497


In [13]:
sold_df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
542,"Carpet,Laminate",False,NaN,NaN,False,675000.0,554248930,roseb7@hotmail.com,2023-07-03,500000.0,...,NaN,NaN,90044,NaN,5988.0,NaN,False,False,NaN,NaN
955,"Laminate,SeeRemarks,Tile",False,NaN,NaN,False,315000.0,552093317,isabels.re4u@gmail.com,2023-10-27,331000.0,...,3.0,NaN,92392,NaN,7525.0,NaN,False,False,NaN,NaN
1057,"Carpet,Laminate,Tile,Wood",True,NaN,NaN,False,375000.0,552062457,LoniVogler@gmail.com,2023-07-05,410000.0,...,2.0,San Jacinto Unified,92582,0.0,7405.0,NaN,False,False,NaN,NaN
1421,NaN,False,NaN,NaN,False,484900.0,551932712,offers@riversidereo1.com,2023-07-19,519000.0,...,2.0,Compton Unified,90221,0.0,4764.0,NaN,False,False,NaN,NaN
2082,"Carpet,Tile,Wood",True,NaN,NaN,False,2740000.0,544556015,gerri@gerriwulffestates.com,2023-06-26,2999822.9,...,3.0,Bonita Unified,91773,671.0,22215.6,NaN,False,False,NaN,NaN


## Feature Selection

The objective of this project is to build a Linear Regression Automated
Valuation Model (AVM) that predicts the final sale price of a property.

The **y variable (prediction)** for this model is:

- ClosePrice

Following the project guidance, the following variables are excluded because
they introduce target leakage or are unavailable for off-market properties:

- ListPrice
- OriginalListPrice

Identifier columns and other non-predictive fields (such as unique IDs and
email addresses) are also removed because they do not contribute meaningful
information for predicting property values.

The remaining property characteristics, location information, and other
intrinsic property attributes will be retained as candidate predictor
variables (X variables) for preprocessing and model development.

In [15]:
# Define the prediction (y variable)
y_variable = "ClosePrice"

print(f"Prediction (y variable): {y_variable}")

Prediction (y variable): ClosePrice


## Feature Selection (X Variables)

The feature variables (X variables) consist of intrinsic property
characteristics and location attributes that are available for both
on-market and off-market properties.

The prediction variable (ClosePrice), listing price variables, identifier
columns, and other non-predictive fields are removed before model training.

In [16]:
drop_columns = [
    "ClosePrice",      # y variable
    "ListPrice",
    "OriginalListPrice",
    "ListingKey",
    "BuyerAgentEmail",
    "ListAgentEmail"
]

model_df = sold_df.drop(columns=drop_columns, errors="ignore")

print(f"Number of features: {model_df.shape[1]}")
model_df.head()

Number of features: 77


,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,CloseDate,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
542,"Carpet,Laminate",False,NaN,NaN,False,2023-07-03,Rose,Belisle,33.974378,-118.295831,...,NaN,NaN,90044,NaN,5988.0,NaN,False,False,NaN,NaN
955,"Laminate,SeeRemarks,Tile",False,NaN,NaN,False,2023-10-27,Isabel,Hernandez,34.491896,-117.374508,...,3.0,NaN,92392,NaN,7525.0,NaN,False,False,NaN,NaN
1057,"Carpet,Laminate,Tile,Wood",True,NaN,NaN,False,2023-07-05,Loni,Vogler,33.782033,-116.993451,...,2.0,San Jacinto Unified,92582,0.0,7405.0,NaN,False,False,NaN,NaN
1421,NaN,False,NaN,NaN,False,2023-07-19,MARIA,NAVARRO,33.895115,-118.195298,...,2.0,Compton Unified,90221,0.0,4764.0,NaN,False,False,NaN,NaN
2082,"Carpet,Tile,Wood",True,NaN,NaN,False,2023-06-26,Gerri,Wulff,34.135025,-117.809712,...,3.0,Bonita Unified,91773,671.0,22215.6,NaN,False,False,NaN,NaN


## Separate X Features and y Variable

The cleaned dataset is separated into:

- X: feature variables
- y: prediction variable (ClosePrice)

These datasets will be used to train and evaluate the Linear Regression model.

In [17]:
X = model_df
y = sold_df.loc[X.index, "ClosePrice"]

print("Feature matrix (X):", X.shape)
print("Prediction variable (y):", y.shape)

Feature matrix (X): (385497, 77)
Prediction variable (y): (385497,)


## Export Cleaned Dataset

In [18]:
model_df.to_csv("cleaned_california_sales.csv", index=False)

print("Cleaned dataset exported successfully.")
print(f"Rows: {model_df.shape[0]}")
print(f"Columns: {model_df.shape[1]}")

Cleaned dataset exported successfully.
Rows: 385497
Columns: 77
